In [77]:
"""
=============================================================================
ENSEMBLE METHODS TUTORIAL
=============================================================================

WHAT ARE ENSEMBLE METHODS?
--------------------------
Ensemble methods combine multiple models to create a stronger predictor than 
any individual model. The idea is that different models might capture different 
patterns in the data, and combining them can reduce errors and improve robustness.

TYPES OF ENSEMBLE METHODS:
-------------------------

1. VOTING (Simple Average)
   - Combines predictions by averaging
   - Each model has equal weight
   - Formula: prediction = (model1 + model2) / 2

2. WEIGHTED VOTING  
   - Combines predictions with different weights
   - Better models get higher weights
   - Formula: prediction = w1*model1 + w2*model2 (where w1+w2=1)

3. BAGGING (Bootstrap Aggregating)
   - Creates multiple versions of the same model type
   - Trains on different subsets of data
   - Averages the results

4. RANDOM FOREST (for Time Series)
   - Uses multiple decision trees
   - Each tree trained on random subsets
   - Good for capturing non-linear patterns

FOR TIME SERIES SPECIFICALLY:
----------------------------
- We'll adapt these methods for sequential data
- Use sliding windows to create features
- Combine SARIMA (linear patterns) + Prophet (trends/seasonality)

"""

"\n=============================================================================\nENSEMBLE METHODS TUTORIAL\n=============================================================================\n\nWHAT ARE ENSEMBLE METHODS?\n--------------------------\nEnsemble methods combine multiple models to create a stronger predictor than \nany individual model. The idea is that different models might capture different \npatterns in the data, and combining them can reduce errors and improve robustness.\n\nTYPES OF ENSEMBLE METHODS:\n-------------------------\n\n1. VOTING (Simple Average)\n   - Combines predictions by averaging\n   - Each model has equal weight\n   - Formula: prediction = (model1 + model2) / 2\n\n2. WEIGHTED VOTING  \n   - Combines predictions with different weights\n   - Better models get higher weights\n   - Formula: prediction = w1*model1 + w2*model2 (where w1+w2=1)\n\n3. BAGGING (Bootstrap Aggregating)\n   - Creates multiple versions of the same model type\n   - Trains on different s

In [78]:
#IMPORTAÇÕES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Time series models
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

# Ensemble methods
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from statsmodels.tools.eval_measures import rmse

# Utilities
from datetime import datetime
import os
import json

In [79]:
print("=== ENSEMBLE METHODS TUTORIAL ===")
print("Combining SARIMA and Prophet for Better Predictions\n")

# Load your prepared data (same as before)
print("=== CARREGANDO DADOS PREPARADOS ===")

# [Insert your data loading code here - same as SARIMA/Prophet]
# This should result in df_final with all installations

# For this tutorial, let's assume df_final is loaded
# df_final should have columns: ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']

=== ENSEMBLE METHODS TUTORIAL ===
Combining SARIMA and Prophet for Better Predictions

=== CARREGANDO DADOS PREPARADOS ===


In [80]:
# ============================================================================
# STEP 1: LOAD BEST PARAMETERS FROM PREVIOUS ANALYSES
# ============================================================================

def load_best_parameters():
    """Load the best parameters from SARIMA and Prophet analyses"""
    best_params = {}
    
    installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
    
    for installation in installations:
        best_params[installation] = {}
        
        # Load SARIMA results
        sarima_file = f"sarima_results_{installation}.csv"
        if os.path.exists(sarima_file):
            sarima_results = pd.read_csv(sarima_file)
            sarima_success = sarima_results[sarima_results['Status'] == 'Success']
            if len(sarima_success) > 0:
                best_sarima = sarima_success.sort_values('MAPE').iloc[0]
                best_params[installation]['sarima'] = {
                    'order': (int(best_sarima['p']), int(best_sarima['d']), int(best_sarima['q'])),
                    'seasonal_order': (int(best_sarima['P']), int(best_sarima['D']), int(best_sarima['Q']), 12),
                    'mape': best_sarima['MAPE']
                }
        
        # Load Prophet results
        prophet_file = f"prophet_results_{installation}.csv"
        if os.path.exists(prophet_file):
            prophet_results = pd.read_csv(prophet_file)
            prophet_success = prophet_results[prophet_results['Status'] == 'Success']
            if len(prophet_success) > 0:
                best_prophet = prophet_success.sort_values('MAPE').iloc[0]
                best_params[installation]['prophet'] = {
                    'yearly_seasonality': int(best_prophet['yearly_seasonality']),
                    'mape': best_prophet['MAPE']
                }
    
    return best_params

In [81]:
# ============================================================================
# STEP 2: INDIVIDUAL MODEL PREDICTIONS
# ============================================================================

def get_sarima_predictions(data, order, seasonal_order):
    """Get SARIMA predictions for the data"""
    try:
        model = SARIMAX(
            data,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False, maxiter=100)
        predictions = results.predict(start=0, end=len(data)-1, dynamic=False, typ="levels")
        return predictions
    except Exception as e:
        print(f"SARIMA Error: {e}")
        return pd.Series([np.nan] * len(data), index=data.index)

def get_prophet_predictions(data, yearly_seasonality):
    """Get Prophet predictions for the data"""
    try:
        # Prepare data for Prophet
        prophet_data = data.reset_index()
        prophet_data.columns = ['ds', 'y']
        prophet_data = prophet_data[prophet_data['y'] > 0]  # Remove zeros
        
        # Train model
        m = Prophet(growth='flat', yearly_seasonality=yearly_seasonality)
        m.fit(prophet_data)
        
        # Make predictions
        future = m.make_future_dataframe(periods=0)
        forecast = m.predict(future)
        
        # Align with original data
        predictions = pd.Series(forecast['yhat'].values, index=data.index)
        return predictions
    except Exception as e:
        print(f"Prophet Error: {e}")
        return pd.Series([np.nan] * len(data), index=data.index)


In [82]:
# ============================================================================
# STEP 3: ENSEMBLE METHOD 1 - SIMPLE VOTING (AVERAGING)
# ============================================================================

def simple_voting_ensemble(sarima_pred, prophet_pred):
    """
    Simple Voting: Average of both predictions
    
    This is the most basic ensemble method.
    Each model contributes equally to the final prediction.
    """
    return (sarima_pred + prophet_pred) / 2

In [83]:
# ============================================================================
# STEP 4: ENSEMBLE METHOD 2 - WEIGHTED VOTING
# ============================================================================

def weighted_voting_ensemble(sarima_pred, prophet_pred, sarima_weight=0.5):
    """
    Weighted Voting: Weighted average based on model performance
    
    Better performing models (lower MAPE) get higher weights.
    """
    prophet_weight = 1 - sarima_weight
    return sarima_weight * sarima_pred + prophet_weight * prophet_pred

def calculate_optimal_weights(sarima_mape, prophet_mape):
    """Calculate optimal weights based on inverse MAPE"""
    if pd.isna(sarima_mape) or pd.isna(prophet_mape):
        return 0.5  # Equal weights if we don't have performance data
    
    # Inverse MAPE weighting (lower MAPE = higher weight)
    sarima_inv = 1 / sarima_mape if sarima_mape > 0 else 1
    prophet_inv = 1 / prophet_mape if prophet_mape > 0 else 1
    
    total_inv = sarima_inv + prophet_inv
    sarima_weight = sarima_inv / total_inv
    
    return sarima_weight

In [84]:
# ============================================================================
# STEP 5: ENSEMBLE METHOD 3 - BAGGING
# ============================================================================

def create_time_series_features(data, window_sizes=[3, 6, 12]):
    """
    Create features for time series regression
    
    This transforms time series into supervised learning problem:
    - Lagged values (previous months)
    - Rolling averages
    - Trend features
    """
    features = pd.DataFrame(index=data.index)
    
    # Add lagged values
    for lag in range(1, 13):  # 12 months of lags
        features[f'lag_{lag}'] = data.shift(lag)
    
    # Add rolling averages
    for window in window_sizes:
        features[f'rolling_mean_{window}'] = data.rolling(window=window).mean()
        features[f'rolling_std_{window}'] = data.rolling(window=window).std()
    
    # Add trend features
    features['month'] = data.index.month
    features['quarter'] = data.index.quarter
    features['year'] = data.index.year
    features['time_index'] = range(len(data))
    
    # Add SARIMA and Prophet predictions as features
    return features

def bagging_ensemble(data, sarima_pred, prophet_pred, n_estimators=10):
    """
    Bagging Ensemble: Multiple models trained on different data subsets
    
    Creates multiple regression models, each trained on a random subset
    of the data, then averages their predictions.
    """
    # Create features
    features = create_time_series_features(data)
    features['sarima_pred'] = sarima_pred
    features['prophet_pred'] = prophet_pred
    
    # Remove rows with NaN (due to lagged features)
    features = features.dropna()
    target = data.loc[features.index]
    
    if len(features) < 20:  # Need minimum data
        print("Not enough data for bagging, using simple average")
        return simple_voting_ensemble(sarima_pred, prophet_pred)
    
    # Split data for training (use 70% for ensemble training)
    train_size = int(0.7 * len(features))
    X_train = features.iloc[:train_size]
    y_train = target.iloc[:train_size]
    
    # Create bagging regressor
    base_estimator = LinearRegression()
    bagging = BaggingRegressor(
        estimator=base_estimator,
        n_estimators=n_estimators,
        random_state=42,
        max_samples=0.8,  # Use 80% of data for each model
        max_features=0.8   # Use 80% of features for each model
    )
    
    # Train the ensemble
    bagging.fit(X_train, y_train)
    
    # Make predictions for all data
    predictions = bagging.predict(features)
    
    # Create full prediction series (fill NaN for missing values)
    full_predictions = pd.Series(index=data.index, dtype=float)
    full_predictions.loc[features.index] = predictions
    
    # Fill NaN values with simple average
    nan_mask = full_predictions.isna()
    full_predictions[nan_mask] = simple_voting_ensemble(sarima_pred, prophet_pred)[nan_mask]
    
    return full_predictions

In [85]:
# ============================================================================
# STEP 6: ENSEMBLE METHOD 4 - RANDOM FOREST
# ============================================================================

def random_forest_ensemble(data, sarima_pred, prophet_pred, n_estimators=100):
    """
    Random Forest Ensemble: Multiple decision trees
    
    Random Forest is excellent at capturing non-linear relationships
    and interactions between different predictions and features.
    """
    # Create features (same as bagging)
    features = create_time_series_features(data)
    features['sarima_pred'] = sarima_pred
    features['prophet_pred'] = prophet_pred
    
    # Remove rows with NaN
    features = features.dropna()
    target = data.loc[features.index]
    
    if len(features) < 20:
        print("Not enough data for Random Forest, using simple average")
        return simple_voting_ensemble(sarima_pred, prophet_pred)
    
    # Split data for training
    train_size = int(0.7 * len(features))
    X_train = features.iloc[:train_size]
    y_train = target.iloc[:train_size]
    
    # Create Random Forest
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )
    
    # Train the model
    rf.fit(X_train, y_train)
    
    # Make predictions
    predictions = rf.predict(features)
    
    # Create full prediction series
    full_predictions = pd.Series(index=data.index, dtype=float)
    full_predictions.loc[features.index] = predictions
    
    # Fill NaN values with simple average
    nan_mask = full_predictions.isna()
    full_predictions[nan_mask] = simple_voting_ensemble(sarima_pred, prophet_pred)[nan_mask]
    
    return full_predictions

In [86]:
# ============================================================================
# STEP 7: EVALUATION FRAMEWORK
# ============================================================================

def evaluate_ensemble_methods(data, installation_id, best_params):
    """Evaluate all ensemble methods for a given installation"""
    
    print(f"\n{'='*60}")
    print(f"EVALUATING ENSEMBLE METHODS - {installation_id}")
    print(f"{'='*60}")
    
    # Get individual model predictions
    print("Getting individual model predictions...")
    
    # SARIMA predictions
    if 'sarima' in best_params:
        sarima_pred = get_sarima_predictions(
            data, 
            best_params['sarima']['order'],
            best_params['sarima']['seasonal_order']
        )
        sarima_mape = best_params['sarima']['mape']
    else:
        print("No SARIMA parameters found, using naive forecast")
        sarima_pred = data.shift(1).fillna(data.mean())
        sarima_mape = 1.0
    
    # Prophet predictions  
    if 'prophet' in best_params:
        prophet_pred = get_prophet_predictions(
            data,
            best_params['prophet']['yearly_seasonality']
        )
        prophet_mape = best_params['prophet']['mape']
    else:
        print("No Prophet parameters found, using naive forecast")
        prophet_pred = data.shift(1).fillna(data.mean())
        prophet_mape = 1.0
    
    # Calculate evaluation split (same as SARIMA/Prophet)
    half = data.iloc[23:]
    
    # Ensemble predictions
    print("Creating ensemble predictions...")
    
    # 1. Simple Voting
    simple_voting_pred = simple_voting_ensemble(sarima_pred, prophet_pred)
    
    # 2. Weighted Voting
    optimal_weight = calculate_optimal_weights(sarima_mape, prophet_mape)
    weighted_voting_pred = weighted_voting_ensemble(sarima_pred, prophet_pred, optimal_weight)
    
    # 3. Bagging
    bagging_pred = bagging_ensemble(data, sarima_pred, prophet_pred)
    
    # 4. Random Forest
    rf_pred = random_forest_ensemble(data, sarima_pred, prophet_pred)
    
    # Evaluate all methods
    methods = {
        'SARIMA': sarima_pred,
        'Prophet': prophet_pred,
        'Simple_Voting': simple_voting_pred,
        'Weighted_Voting': weighted_voting_pred,
        'Bagging': bagging_pred,
        'Random_Forest': rf_pred
    }
    
    results = []
    
    print(f"\n{'Method':<15} {'MAPE':<8} {'MAE':<10} {'RMSE':<10} {'WMAPE':<8}")
    print("-" * 55)
    
    for method_name, predictions in methods.items():
        # Get predictions for evaluation period
        half_pred = predictions.iloc[23:]
        
        # Calculate metrics
        try:
            mape = mean_absolute_percentage_error(half, half_pred)
            mae = mean_absolute_error(half, half_pred)
            rmse_val = rmse(half, half_pred)
            wmape = mean_absolute_percentage_error(half, half_pred, sample_weight=half)
            
            results.append({
                'Installation': installation_id,
                'Method': method_name,
                'MAPE': mape,
                'MAE': mae,
                'RMSE': rmse_val,
                'WMAPE': wmape,
                'Optimal_Weight': optimal_weight if method_name == 'Weighted_Voting' else None
            })
            
            print(f"{method_name:<15} {mape*100:6.2f}% {mae:8.2f} {rmse_val:8.2f} {wmape*100:6.2f}%")
            
        except Exception as e:
            print(f"{method_name:<15} ERROR: {str(e)[:30]}")
            results.append({
                'Installation': installation_id,
                'Method': method_name,
                'MAPE': np.nan,
                'MAE': np.nan,
                'RMSE': np.nan,
                'WMAPE': np.nan,
                'Optimal_Weight': None
            })
    
    return results

In [87]:
# ============================================================================
# STEP 8: MAIN EXECUTION
# ============================================================================

def run_ensemble_analysis():
    """Run ensemble analysis for all installations"""
    
    # Load best parameters
    print("Loading best parameters from previous analyses...")
    best_params = load_best_parameters()
    
    # Display loaded parameters
    print("\nLoaded Parameters:")
    for installation, params in best_params.items():
        print(f"{installation}:")
        if 'sarima' in params:
            print(f"  SARIMA: {params['sarima']['order']} x {params['sarima']['seasonal_order']} (MAPE: {params['sarima']['mape']*100:.2f}%)")
        if 'prophet' in params:
            print(f"  Prophet: yearly_seasonality={params['prophet']['yearly_seasonality']} (MAPE: {params['prophet']['mape']*100:.2f}%)")
    
    # Run ensemble analysis for each installation
    all_results = []
    installations = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
    
    for installation in installations:
        if installation in df_final.columns and installation in best_params:
            data = df_final[installation]
            results = evaluate_ensemble_methods(data, installation, best_params[installation])
            all_results.extend(results)
        else:
            print(f"Skipping {installation} - no data or parameters")
    
    # Save results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('ensemble_results.csv', index=False)
    
    # Summary
    print(f"\n{'='*80}")
    print("ENSEMBLE ANALYSIS SUMMARY")
    print(f"{'='*80}")
    
    # Best method per installation
    for installation in installations:
        inst_results = results_df[results_df['Installation'] == installation]
        if len(inst_results) > 0:
            best = inst_results.sort_values('MAPE').iloc[0]
            print(f"{installation}: {best['Method']} (MAPE: {best['MAPE']*100:.2f}%)")
    
    # Overall best method
    avg_by_method = results_df.groupby('Method')['MAPE'].mean().sort_values()
    print(f"\nBest Overall Method: {avg_by_method.index[0]} (Avg MAPE: {avg_by_method.iloc[0]*100:.2f}%)")
    
    return results_df

In [88]:
print("=== PROCESSANDO DADOS DO EXCEL ===")
# Lendo os dados do Excel (código original adaptado)
df_excel = pd.read_excel(
    "../Controle_Faturamento_2023.xlsx",
    sheet_name="Consumo Médio - Clientes",
)

# Removendo dados desnecessários (mantendo UC)
df_excel_clean = df_excel.copy()
df_excel_clean.drop(df_excel_clean.iloc[:, 48:68], inplace=True, axis=1)
df_excel_clean.drop("Nº Cliente", inplace=True, axis=1)
df_excel_clean.drop("Cliente", inplace=True, axis=1)

df_excel_clean = df_excel_clean.head()

# Extraindo os UCs que serão usados para filtrar os dados
selected_ucs = df_excel_clean['UC'].astype(str).tolist()
print(f"UCs selecionados do Excel: {selected_ucs}")

# Corrigindo os dados para uso das fórmulas (Colocando a data como índice do dataset)
df_excel_transpose1 = df_excel_clean.transpose()
df_excel_transpose = df_excel_transpose1.rename(columns=df_excel_transpose1.iloc[0])
df_excel_transpose.drop("UC", inplace=True, axis=0)
df_excel_transpose.dropna(inplace=True)

# Transformando índice em datetime para o Excel
some_dates = np.array(df_excel_transpose.index, dtype="datetime64[D]")
idx_excel = pd.DatetimeIndex(some_dates)
df_excel_transpose.index = idx_excel

# Convertendo Excel para formato long (igual ao CSV)
excel_long = []
for installation in df_excel_transpose.columns:
    for date in df_excel_transpose.index:
        value = df_excel_transpose.loc[date, installation]
        if pd.notna(value):  # Só adiciona valores não-nulos
            excel_long.append({
                'Date': date,
                'Installation': str(installation),  # Convertendo para string para compatibilidade
                'ConsumeValue': float(value),
                'Source': 'Excel'
            })

df_excel_long = pd.DataFrame(excel_long)

# Lendo os dados do CSV
df_csv = pd.read_csv("../_SELECT_cb_Name_cb_Installation_fc_ForecastConsumeValue_fc_Refer_202504140945.csv")

# Convertendo a coluna Date para datetime
df_csv['Date'] = pd.to_datetime(df_csv['Date'])
df_csv['Installation'] = df_csv['Installation'].astype(str)  # Garantindo que seja string

# Filtrando CSV para manter apenas os mesmos UCs do Excel
df_csv_filtered = df_csv[df_csv['Installation'].isin(selected_ucs)].copy()
df_csv_filtered['Source'] = 'CSV'

print("\n=== COMBINANDO OS DADOS ===")
# Combinando os dataframes
df_combined = pd.concat([df_excel_long, df_csv_filtered], ignore_index=True)

# Verificando duplicatas na combinação Date + Installation
duplicate_check = df_combined.groupby(['Date', 'Installation']).size()
duplicates = duplicate_check[duplicate_check > 1]

if len(duplicates) > 0:
    print(f"\nEncontradas {len(duplicates)} combinações Date-Installation duplicadas")
    print("Resolvendo duplicatas...")
    
    df_combined_clean = df_combined.sort_values(['Date', 'Installation', 'Source']).drop_duplicates(
        subset=['Date', 'Installation'], keep='last')  # CSV vem depois do Excel na ordenação
    
    print(f"Dados após resolver duplicatas: {len(df_combined_clean)} registros")
else:
    df_combined_clean = df_combined.copy()
    print("Nenhuma duplicata encontrada!")

# Fazendo o pivot com os dados limpos
df_final = df_combined_clean.pivot(index='Date', columns='Installation', values='ConsumeValue')

# Substituindo valores NaN por 0
df_final.fillna(0, inplace=True)

# Ordenando o índice por data
df_final.sort_index(inplace=True)

# Removendo dados a partir de 2025-01-01
df_final = df_final[df_final.index < '2025-01-01']

# Se a frequência não for detectada ou for diferente de mensal, resample para mensal
date_diffs = df_final.index.to_series().diff().dropna()
print(f"Diferenças entre datas: {date_diffs.value_counts().head()}")

# Se os dados já são mensais, tenta inferir a frequência
try:
    df_final.index.freq = pd.infer_freq(df_final.index)
    print(f"Frequência inferida: {df_final.index.freq}")
except:
    print("Não foi possível inferir frequência automaticamente")

# Se a frequência não for detectada ou for diferente de mensal, resample para mensal
if df_final.index.freq != 'MS' and df_final.index.freq != 'M':
    print("Reamostrando dados para frequência mensal...")
    # Agrupa por mês e soma os valores (ou use .mean() se preferir média)
    df_final = df_final.resample('MS').mean()
    df_final.index.freq = 'MS'
    print(f"Nova frequência: {df_final.index.freq}")
    print(f"Novo shape após reamostragem: {df_final.shape}")
else:
    print("Dados já estão em frequência mensal")

df_final.dropna(inplace=True)

print("\n=== ESTRUTURA FINAL DOS DADOS ===")
print("Colunas disponíveis (Installations):")
print(df_final.columns.tolist())
print(f"\nShape dos dados: {df_final.shape}")
print(f"Período dos dados: {df_final.index.min()} até {df_final.index.max()}")

=== PROCESSANDO DADOS DO EXCEL ===
UCs selecionados do Excel: ['3003858507', '3001084033', '3011373971', '3001449459', '3011504476']

=== COMBINANDO OS DADOS ===

Encontradas 10 combinações Date-Installation duplicadas
Resolvendo duplicatas...
Dados após resolver duplicatas: 354 registros
Diferenças entre datas: Date
31 days    35
30 days    20
28 days     5
29 days     2
2 days      2
Name: count, dtype: int64
Frequência inferida: None
Reamostrando dados para frequência mensal...
Nova frequência: <MonthBegin>
Novo shape após reamostragem: (67, 5)

=== ESTRUTURA FINAL DOS DADOS ===
Colunas disponíveis (Installations):
['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']

Shape dos dados: (67, 5)
Período dos dados: 2019-06-01 00:00:00 até 2024-12-01 00:00:00


In [89]:
# ============================================================================
# TUTORIAL EXECUTION
# ============================================================================
    
print("To run this tutorial:")
print("1. Make sure df_final is loaded with your time series data")
print("2. Ensure SARIMA and Prophet result files exist")
print("3. Run: results = run_ensemble_analysis()")
print("4. Check ensemble_results.csv for detailed comparison")
    
# Uncomment to run:
results = run_ensemble_analysis()

To run this tutorial:
1. Make sure df_final is loaded with your time series data
2. Ensure SARIMA and Prophet result files exist
3. Run: results = run_ensemble_analysis()
4. Check ensemble_results.csv for detailed comparison
Loading best parameters from previous analyses...

Loaded Parameters:
3001084033:
  SARIMA: (9, 1, 9) x (1, 0, 1, 12) (MAPE: 8.24%)
  Prophet: yearly_seasonality=12 (MAPE: 13.73%)
3001449459:
  SARIMA: (0, 0, 0) x (0, 0, 0, 12) (MAPE: 97.73%)
  Prophet: yearly_seasonality=12 (MAPE: 6697820881001806848.00%)
3003858507:
  SARIMA: (10, 1, 9) x (1, 0, 1, 12) (MAPE: 18.01%)
  Prophet: yearly_seasonality=12 (MAPE: 39.34%)
3011373971:
  SARIMA: (0, 0, 0) x (3, 0, 10, 12) (MAPE: 97.73%)
  Prophet: yearly_seasonality=11 (MAPE: 2492860400021801984.00%)
3011504476:
  SARIMA: (0, 0, 0) x (1, 0, 10, 12) (MAPE: 93.18%)
  Prophet: yearly_seasonality=12 (MAPE: 28576990395380289536.00%)

EVALUATING ENSEMBLE METHODS - 3001084033
Getting individual model predictions...


15:25:11 - cmdstanpy - INFO - Chain [1] start processing
15:25:11 - cmdstanpy - INFO - Chain [1] done processing
15:25:11 - cmdstanpy - INFO - Chain [1] start processing


Creating ensemble predictions...

Method          MAPE     MAE        RMSE       WMAPE   
-------------------------------------------------------
SARIMA            8.24%   478.46   675.88   7.38%
Prophet          13.73%   708.05   929.10  10.92%
Simple_Voting     9.47%   501.08   644.51   7.73%
Weighted_Voting   8.72%   468.04   616.34   7.22%
Bagging           9.46%   404.92   712.38   6.25%
Random_Forest     9.50%   442.86   677.86   6.83%

EVALUATING ENSEMBLE METHODS - 3001449459
Getting individual model predictions...


15:25:11 - cmdstanpy - INFO - Chain [1] done processing


Prophet Error: Length of values (65) does not match length of index (67)
Creating ensemble predictions...
Not enough data for bagging, using simple average
Not enough data for Random Forest, using simple average

Method          MAPE     MAE        RMSE       WMAPE   
-------------------------------------------------------
SARIMA           97.73%  1005.55  1108.59 100.00%
Prophet         ERROR: Input contains NaN.
Simple_Voting   ERROR: Input contains NaN.
Weighted_Voting ERROR: Input contains NaN.
Bagging         ERROR: Input contains NaN.
Random_Forest   ERROR: Input contains NaN.

EVALUATING ENSEMBLE METHODS - 3003858507
Getting individual model predictions...


...

Creating ensemble predictions...

Method          MAPE     MAE        RMSE       WMAPE   
-------------------------------------------------------
SARIMA           18.01%   347.59   447.97  15.94%
Prophet          39.34%   604.24   728.47  27.70%
Simple_Voting    23.25%   390.16   483.70  17.89%
Weighted_Voting  19.02%   346.05   436.82  15.87%
Bagging          10.59%   182.88   230.17   8.38%
Random_Forest    15.35%   269.04   338.80  12.33%

EVALUATING ENSEMBLE METHODS - 3011373971
Getting individual model predictions...


15:25:14 - cmdstanpy - INFO - Chain [1] start processing
15:25:14 - cmdstanpy - INFO - Chain [1] done processing


Prophet Error: Length of values (61) does not match length of index (67)
Creating ensemble predictions...
Not enough data for bagging, using simple average
Not enough data for Random Forest, using simple average

Method          MAPE     MAE        RMSE       WMAPE   
-------------------------------------------------------
SARIMA           97.73%   573.34   599.58 100.00%
Prophet         ERROR: Input contains NaN.
Simple_Voting   ERROR: Input contains NaN.
Weighted_Voting ERROR: Input contains NaN.
Bagging         ERROR: Input contains NaN.
Random_Forest   ERROR: Input contains NaN.

EVALUATING ENSEMBLE METHODS - 3011504476
Getting individual model predictions...


15:25:15 - cmdstanpy - INFO - Chain [1] start processing
15:25:15 - cmdstanpy - INFO - Chain [1] done processing


Prophet Error: Length of values (64) does not match length of index (67)
Creating ensemble predictions...
Not enough data for bagging, using simple average
Not enough data for Random Forest, using simple average

Method          MAPE     MAE        RMSE       WMAPE   
-------------------------------------------------------
SARIMA           93.18%  1428.86  1641.91 100.00%
Prophet         ERROR: Input contains NaN.
Simple_Voting   ERROR: Input contains NaN.
Weighted_Voting ERROR: Input contains NaN.
Bagging         ERROR: Input contains NaN.
Random_Forest   ERROR: Input contains NaN.

ENSEMBLE ANALYSIS SUMMARY
3001084033: SARIMA (MAPE: 8.24%)
3001449459: SARIMA (MAPE: 97.73%)
3003858507: Bagging (MAPE: 10.59%)
3011373971: SARIMA (MAPE: 97.73%)
3011504476: SARIMA (MAPE: 93.18%)

Best Overall Method: Bagging (Avg MAPE: 10.03%)
